# 9장 웹부록 | 가상 세계에서 걷기를 배우는 로봇

「영화로 보는 AI 리터러시」 9장 '피지컬 AI: 세상 속으로 걸어 나온 AI'의 코랩 실습입니다.

**쓰는 법**

- 코드를 몰라도 괜찮습니다. 위에서부터 차례로 셀 왼쪽의 ▶(실행) 버튼을 누르세요.
- `# ← 이 숫자를 바꿔 보세요` 표시가 있는 곳만 고치고, 그 셀부터 다시 실행해 보세요.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 따로 설치할 것은 없습니다. 그래프 글자는 영어로 나오고, 설명은 이 글에 우리말로 적었습니다.

**이 실습에서 할 일**

1. 컴퓨터 속 가상 세계에 아주 단순한 다리 로봇을 만듭니다.
2. 강화학습으로 걷기를 연습시킵니다. 보상을 '앞으로 간 거리'로만 줄 때와 '넘어지면 벌점'을 더할 때를 비교합니다.
3. 바닥의 마찰을 무작위로 바꿔 가며 연습시킨 로봇(도메인 무작위화)이 미끄러운 바닥에서 얼마나 잘 버티는지 확인합니다.

전체를 실행하는 데 1분 안쪽이 걸립니다.

## 1단계. 가상 세계와 로봇 만들기

진짜 물리 엔진 대신, 핵심만 남긴 작은 시뮬레이터를 직접 만듭니다. 우리 로봇은 이렇게 생겼습니다.

- 몸통은 길이 1m짜리 막대이고, 아래에 발이 하나 있습니다. 발로 땅을 앞뒤로 밀어 움직입니다.
- 몸통이 앞으로 기울면 넘어지려 합니다. 그래서 발을 앞으로 내밀어 몸을 받쳐야 합니다. 사람이 걸을 때 '넘어지려는 몸을 다리로 받치는' 것과 같습니다.
- 몸통이 약 30도 넘게 기울면 **넘어진 것**으로 보고 그 자리에서 멈춥니다.
- 발이 땅을 미는 힘은 **바닥의 마찰**보다 셀 수 없습니다. 마찰이 작으면(빙판) 발이 미끄러집니다.
- 빨리 걸을수록 발이 땅에 세게 닿아 몸이 더 많이 흔들립니다.

로봇의 '두뇌'는 숫자 4개입니다. 이 숫자들이 '기울기, 기우는 속도, 걷는 속도를 보고 발을 얼마나 세게 밀지'를 정합니다. 3장에서 본 가중치와 같은 것입니다. 이 4개의 숫자를 잘 고르는 것이 이 실습의 '학습'입니다.

아래 셀을 실행하면 시뮬레이터가 준비됩니다. 한 번에 로봇 수백 대를 동시에 돌립니다. 책 3-4절에서 본 '가상 세계에서 수천 대가 동시에 연습하기'를 작게 해 보는 것입니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

G, LEG, DT, STEPS = 9.8, 1.0, 0.04, 250   # 중력, 몸통 길이(m), 0.04초 간격, 250번 = 10초
FALL_ANGLE = 0.5                           # 약 30도 넘게 기울면 넘어진 것으로 봅니다

def simulate(brains, friction=1.0, seed=0, record=False):
    """로봇 여러 대를 동시에 10초 동안 걷게 합니다.
    brains: 로봇마다 숫자 4개짜리 두뇌 / friction: 바닥 마찰(로봇마다 달라도 됩니다)"""
    brains = np.atleast_2d(brains)
    n = len(brains)
    rng = np.random.default_rng(seed)
    mu = np.broadcast_to(np.asarray(friction, float), (n,)).copy()
    x = np.zeros(n); v = np.zeros(n)                 # 위치, 걷는 속도
    tilt = rng.normal(0, 0.03, n); spin = np.zeros(n)  # 몸통 기울기, 기우는 속도
    alive = np.ones(n, bool)
    log = []
    for t in range(STEPS):
        # 1) 판단: 두뇌가 '발을 얼마나 세게 밀지' 정합니다 (-1 ~ +1)
        push = np.tanh(brains[:, 0]*tilt + brains[:, 1]*spin + brains[:, 2]*v + brains[:, 3])
        # 2) 행동: 발이 땅을 밉니다. 마찰보다 세게는 못 밉니다(미끄러짐)
        force = np.clip(15.0 * push, -mu * G, mu * G)
        # 3) 물리: 중력은 몸을 넘어뜨리고, 발을 미는 힘은 몸을 반대로 세웁니다
        tilt_acc = (G * np.sin(tilt) - force * np.cos(tilt)) / LEG
        shake = rng.normal(0, 1, n) * (0.15 + 0.4 * np.abs(v)) * np.sqrt(DT)  # 빠를수록 더 흔들림
        spin = np.where(alive, spin + tilt_acc * DT + shake, 0)
        tilt = np.where(alive, tilt + spin * DT, tilt)
        v = np.where(alive, v + force * DT, 0)
        x = np.where(alive, x + v * DT, x)
        alive &= ~(np.abs(tilt) > FALL_ANGLE)       # 4) 감지: 넘어졌는지 확인
        if record:
            log.append((x.copy(), tilt.copy(), alive.copy()))
    fell = ~alive
    return (x, fell, log) if record else (x, fell)

print("시뮬레이터 준비 완료!")

### 아직 아무것도 배우지 않은 로봇

먼저 두 로봇을 봅니다.

- **아기 로봇**: 넘어지지 않게 버티는 법만 조금 압니다. 두뇌 = `[3, 1, 0, 0]`
- **엉터리 로봇**: 두뇌 숫자를 아무렇게나 정했습니다.

그림은 0.5초마다 찍은 로봇의 모습입니다. 막대가 몸통, 아래 점이 발입니다. 빨간색은 넘어진 모습입니다.

In [ ]:
def draw_robot(ax, brain, title, friction=1.0, seed=3):
    x, fell, log = simulate(brain, friction, seed=seed, record=True)
    for t in range(0, STEPS, 12):
        xs, ts, al = log[t]
        foot, head = xs[0], (xs[0] + LEG*np.sin(ts[0]), LEG*np.cos(ts[0]))
        color = "tab:blue" if al[0] else "tab:red"
        ax.plot([foot, head[0]], [0, head[1]], color=color, alpha=0.3 + 0.7*t/STEPS, lw=2)
        ax.plot(foot, 0, "o", color=color, ms=4)
    ax.axhline(0, color="gray", lw=1)
    ax.set_title(f"{title}: {x[0]:.1f} m, {'FELL' if fell[0] else 'standing'}")
    ax.set_ylim(-0.1, 1.2); ax.set_xlabel("position (m)")

baby = np.array([3.0, 1.0, 0.0, 0.0])
random_brain = np.random.default_rng(1).normal(0, 2, 4)

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5))
draw_robot(axes[0], baby, "Baby robot")
draw_robot(axes[1], random_brain, "Random brain")
plt.tight_layout(); plt.show()

아기 로봇은 서 있기는 하지만 거의 앞으로 가지 않습니다. 엉터리 로봇은 금방 넘어집니다. 이제 걷기를 가르쳐 보겠습니다.

## 2단계. 강화학습으로 걷기 연습시키기: 보상 = 앞으로 간 거리

책 7장과 9장 3-3절의 **강화학습**은 '잘하면 점수, 못하면 벌점'을 주며 스스로 연습하게 하는 방법입니다. 여기서는 가장 단순한 방법을 씁니다.

1. 지금 두뇌의 숫자를 여러 방향으로 **조금씩 흔들어 본** 두뇌들을 만듭니다.
2. 각 두뇌로 로봇 수십 대를 10초씩 걷게 하고 **보상**(점수)을 잽니다.
3. 점수가 오른 쪽으로 두뇌를 조금 옮깁니다. 이것을 수십 번 되풀이합니다.

한 번 연습할 때마다 로봇 480대가 가상 세계에서 동시에 걷습니다. 먼저 보상을 **'앞으로 간 거리'로만** 줍니다. 넘어져도 따로 벌점은 없습니다. 넘어지면 그 뒤로 못 가니 손해이긴 합니다.

In [ ]:
def train(fall_penalty=0.0, friction_range=None, rounds=80, seed=0):
    """두뇌 숫자 4개를 연습으로 고칩니다.
    fall_penalty: 넘어지면 빼는 점수 / friction_range: (최소, 최대)를 주면 연습마다 바닥 마찰을 무작위로 바꿉니다"""
    rng = np.random.default_rng(seed)
    brain = baby.copy()
    history = []
    for r in range(rounds):
        tries = rng.normal(0, 1, (8, 4))                       # 흔들어 볼 방향 8개
        cand = np.concatenate([brain + 0.3*tries, brain - 0.3*tries])
        brains = np.repeat(cand, 30, axis=0)                   # 두뇌 하나당 로봇 30대
        if friction_range is None:
            fr = 1.0
        else:
            fr = rng.uniform(friction_range[0], friction_range[1], len(brains))
        x, fell = simulate(brains, fr, seed=1000 + r)
        reward = (x - fall_penalty * fell).reshape(16, 30).mean(axis=1)
        up, down = reward[:8], reward[8:]
        brain = brain + 0.09 * ((up - down)[:, None] * tries).mean(axis=0) / (reward.std() + 1e-6)
        history.append(reward.mean())
    return brain, history

def test(brain, friction=1.0, n=500, seed=7):
    """처음 보는 상황에서 로봇 500대를 걷게 하고 평균 거리와 넘어진 비율을 잽니다."""
    x, fell = simulate(np.tile(brain, (n, 1)), friction, seed=seed)
    return x.mean(), fell.mean()

brain_distance, hist_distance = train(fall_penalty=0.0)
d, f = test(brain_distance)
print(f"[보상 = 거리만]  평균 {d:.1f} m 전진, 넘어진 비율 {f:.0%}")

## 3단계. '넘어지면 벌점'을 더하기

이번에는 같은 연습을 하되, 넘어지면 점수를 크게 깎습니다. 벌점은 아래 `FALL_PENALTY`로 정합니다. 20이면 '넘어지는 것은 20m 덜 간 것만큼 나쁘다'는 뜻입니다.

👉 **바꿔 볼 곳**: `FALL_PENALTY`를 `0`, `5`, `20`, `50`으로 바꿔 가며 실행해 보세요. 거리와 넘어진 비율이 어떻게 바뀝니까?

In [ ]:
FALL_PENALTY = 20   # ← 이 숫자를 바꿔 보세요 (넘어지면 깎는 점수)

brain_penalty, hist_penalty = train(fall_penalty=FALL_PENALTY)

rows = []
for name, b in [("아기 로봇(연습 전)", baby), ("보상 = 거리만", brain_distance),
                (f"보상 = 거리 - 넘어짐×{FALL_PENALTY}", brain_penalty)]:
    d, f = test(b)
    rows.append({"로봇": name, "평균 전진 거리(m)": round(d, 1), "넘어진 비율": f"{f:.0%}"})
display(pd.DataFrame(rows))

def draw_runs(ax, brain, title, n=20):
    """로봇 20대가 시간에 따라 어디까지 갔는지 그립니다. 넘어진 로봇은 빨간 X에서 멈춥니다."""
    x, fell, log = simulate(np.tile(brain, (n, 1)), 1.0, seed=11, record=True)
    pos = np.array([l[0] for l in log]); alive = np.array([l[2] for l in log])
    t = np.arange(STEPS) * DT
    for i in range(n):
        ax.plot(t, pos[:, i], color="tab:red" if fell[i] else "tab:blue", alpha=0.6)
        if fell[i]:
            k = np.argmin(alive[:, i]); ax.plot(t[k], pos[k, i], "x", color="tab:red")
    ax.set_title(f"{title}: {fell.sum()} of {n} fell"); ax.set_xlabel("time (s)"); ax.set_ylabel("position (m)")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(hist_distance, label="reward = distance")
axes[0].plot(hist_penalty, label=f"reward = distance - {FALL_PENALTY} x fall")
axes[0].set_xlabel("practice round"); axes[0].set_ylabel("average reward"); axes[0].legend()
axes[0].set_title("Learning curves")
draw_runs(axes[1], brain_distance, "Distance only")
draw_runs(axes[2], brain_penalty, "With fall penalty")
plt.tight_layout(); plt.show()

**읽는 법**: 가운데와 오른쪽 그림의 선 하나가 로봇 한 대입니다. 선이 위로 올라갈수록 멀리 간 것이고, 빨간 X는 그 자리에서 넘어졌다는 뜻입니다. '거리만' 로봇은 멀리 가지만 꽤 자주 넘어집니다. 빨리 걸을수록 많이 흔들리는데, 점수표에는 넘어짐이 크게 손해로 적히지 않으니 위험을 무릅쓰고 서두르는 것입니다. 벌점을 더하면 조금 덜 가는 대신 훨씬 덜 넘어집니다.

로봇은 우리가 **원한 것**이 아니라 우리가 **점수로 적은 것**을 배웁니다. 진짜 로봇이라면 넘어질 때마다 부서지니, 보상을 어떻게 적느냐가 곧 로봇의 성격이 됩니다.

## 4단계. 처음 보는 바닥에 세워 보기: 현실 격차

지금까지 연습한 바닥은 모두 마찰이 1.0인 '마른 운동장'이었습니다. 진짜 세상의 바닥은 다릅니다. 물 묻은 타일, 먼지 쌓인 마루, 살얼음이 있습니다. 마찰을 0.2(아주 미끄러움)부터 1.2(고무 매트)까지 바꿔 가며 3단계의 로봇을 시험해 봅니다.

이렇게 가상에서 잘하던 로봇이 현실에서 넘어지는 차이를 책에서는 **현실 격차**(reality gap)라고 불렀습니다.

In [ ]:
frictions = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2]
fall_fixed = [test(brain_penalty, mu)[1] for mu in frictions]
for mu, f in zip(frictions, fall_fixed):
    print(f"마찰 {mu:.1f}  →  넘어진 비율 {f:.0%}")

마찰 1.0에서 연습한 로봇은 바닥이 조금만 미끄러워져도 크게 흔들립니다. 연습한 세상 하나만 알기 때문입니다.

## 5단계. 도메인 무작위화: 연습할 때마다 바닥을 바꾸기

책의 '심화 상식' 상자에서 본 **도메인 무작위화**를 해 봅니다. 연습할 때마다 로봇마다 바닥 마찰을 무작위로 정합니다. 어떤 로봇은 빙판에서, 어떤 로봇은 고무 매트에서 연습합니다. 벌점은 3단계와 같게 둡니다.

👉 **바꿔 볼 곳**: 연습 바닥의 범위 `FRICTION_MIN`, `FRICTION_MAX`를 바꿔 보세요. 범위를 너무 넓히면 어떻게 됩니까?

In [ ]:
FRICTION_MIN = 0.3   # ← 이 숫자를 바꿔 보세요 (가장 미끄러운 연습 바닥)
FRICTION_MAX = 1.2   # ← 이 숫자를 바꿔 보세요 (가장 덜 미끄러운 연습 바닥)

brain_random, hist_random = train(fall_penalty=FALL_PENALTY, friction_range=(FRICTION_MIN, FRICTION_MAX))
fall_random = [test(brain_random, mu)[1] for mu in frictions]

table = pd.DataFrame({
    "바닥 마찰": frictions,
    "마찰 1.0에서만 연습: 넘어진 비율": [f"{f:.0%}" for f in fall_fixed],
    "무작위 바닥에서 연습: 넘어진 비율": [f"{f:.0%}" for f in fall_random],
})
display(table)

plt.figure(figsize=(7, 4))
plt.plot(frictions, np.array(fall_fixed)*100, "o-", label="trained on friction 1.0 only")
plt.plot(frictions, np.array(fall_random)*100, "s-", label=f"trained on random friction {FRICTION_MIN}-{FRICTION_MAX}")
plt.axvspan(FRICTION_MIN, FRICTION_MAX, color="gray", alpha=0.1)
plt.xlabel("floor friction (low = slippery)"); plt.ylabel("robots that fell (%)")
plt.title("Domain randomization makes the robot sturdier"); plt.legend(); plt.show()

d1, _ = test(brain_penalty); d2, _ = test(brain_random)
print(f"마른 바닥(1.0)에서 평균 거리: 한 바닥만 연습 {d1:.1f} m / 무작위 바닥 연습 {d2:.1f} m")

**읽는 법**: 무작위 바닥에서 연습한 로봇은 미끄러운 바닥에서 훨씬 덜 넘어집니다. 대신 마른 바닥에서는 조금 덜 갑니다. 어떤 바닥이 나와도 버티도록 조심스럽게 걷는 법을 배웠기 때문입니다. 튼튼함과 빠르기 사이에 맞바꿈이 있습니다.

## 더 해 보기

- 3단계의 `FALL_PENALTY`를 아주 크게(예: 200) 하면 로봇이 아예 걷지 않으려 하는지 확인해 보세요. 왜 그럴까요?
- 5단계의 `FRICTION_MIN`을 `0.6`으로 올리면 빙판(0.2~0.3)에서 어떻게 됩니까? '연습에서 본 적 없는 세상'의 범위를 생각해 보세요.
- `simulate` 안의 `0.4 * np.abs(v)`를 `0.0`으로 바꾸면 '빨리 걸어도 흔들리지 않는 세상'이 됩니다. 이때 두 보상의 차이는 어떻게 됩니까?

## 생각해 볼 것

1. '거리만' 보상을 받은 로봇은 자주 넘어지면서도 서둘렀습니다. 7장에서 본 '점수를 올리는 엉뚱한 지름길'과 어떤 점이 닮았습니까? 진짜 로봇 회사라면 보상에 무엇을 더 넣겠습니까?
2. 이 실습의 로봇은 가상 세계에서 10초씩 수만 번 걸었습니다. 진짜 로봇으로 같은 연습을 하면 무엇이 문제가 될까요? 책 3-4절의 '시뮬레이션에서 배우는 이유'와 연결해 쓰세요.
3. 도메인 무작위화로 바닥 마찰은 대비했습니다. 진짜 세상에서 로봇을 넘어뜨릴 수 있는 다른 차이(무게, 모터 힘, 계단, 사람과의 부딪힘 등)를 두 가지 들고, 가상 세계에서 어떻게 흔들어 연습시킬지 제안해 보세요.